# 2. Use the framework to compare research ideas

Suppose our Ridge model worked well during development, and we're ready to use it in 2015. Should we keep that fitted model for the whole year, or refit it each quarter as more returns become known?

We'll compare those two choices using the framework from the first notebook. Both models will feed scores into `PredictedScoreStrategy`, while `MomentumStrategy` will calculate a comparison signal from the history supplied by the backtester. They share the monthly schedule and allocation from `MonthlyTopNStrategy`, and the same backtester and OMS handle their trades. That lets us study retraining while keeping the portfolio rules consistent.

This notebook runs independently on the **30-stock class extract**. We'll evaluate 2015 here; HW4 uses the full universe and the April 2016–September 2026 final period.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display
import sys
%matplotlib inline
course_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
modeling_dir = course_root / 'Lectures/modeling'
if str(modeling_dir) not in sys.path:
    sys.path.insert(0, str(modeling_dir))
from course_data import load_data, load_membership, prepare_panel, date_folds, make_pipeline, prediction_metrics, FEATURES
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

# Pick up changes to the shared framework when rerunning this cell.
import importlib
import backtest
importlib.reload(backtest)

from backtest import (MomentumStrategy, PredictedScoreStrategy, OrderManagementSystem,
                      Backtester, BacktestAnalytics)

In [ ]:
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
import sklearn
import duckdb
import json
prices, fundamentals, universe, benchmark = load_data()
membership = load_membership()
panel = prepare_panel()
calendar = pd.DatetimeIndex(benchmark.date)
fit_date = pd.Timestamp('2014-12-31')
development = panel.loc[panel.eligible & panel.date.ge('2010-01-01')
                        & panel.date.lt(fit_date) & panel.label_end.lt(fit_date)].reset_index(drop=True)
folds = date_folds(development, validation_years=(2012, 2013, 2014))

## 1. Set up the experiment before looking at 2015

Let's put ourselves at the end of 2014. We have several years of features and completed returns, and we want to choose a Ridge penalty before starting the next year's experiment. We'll try three values of `alpha` and choose the one with the lowest mean RMSE across the 2012, 2013, and 2014 validation folds. The 2015 outcomes play no part in that choice.

Once we've chosen the penalty, we'll use the same features, monthly top-three rule, holding period, and costs in both runs. One model will keep its original fit, while the other will update quarterly. Their portfolio results will then reflect the forecasts and trades produced by those two fitting schedules.

The configuration below records these choices along with the data and package versions. Saving them helps another researcher reproduce the run, even if the source database or software later changes.

In [ ]:
experiment = {
    'name': 'class_ridge_2015', 'database': 'equities_class.duckdb',
    'features': FEATURES, 'training_signal_start': '2010-01-01',
    'initial_fit_signal': str(fit_date.date()), 'validation_years': [2012, 2013, 2014],
    'selection_metric': 'mean fold RMSE', 'alpha_grid': [1, 100, 1000],
    'test_start': '2015-01-01', 'test_end': '2015-12-31',
    'top_n': 3, 'cohort_fraction': 1/3, 'holding_calendar_days': 90,
    'fee_bps_per_side': 5, 'initial_cash': 1_000_000,
    'cash_yield': 0, 'risk_free_hurdle': 0.02,
    'universe_policy': 'quarterly membership for next-session entries; never filter held-price history',
    'quarterly_fit_signals': ['2014-12-31', '2015-03-31', '2015-06-30', '2015-09-30'],
    'versions': {'pandas': pd.__version__, 'numpy': np.__version__,
                 'sklearn': sklearn.__version__, 'duckdb': duckdb.__version__},
}
with duckdb.connect(str(modeling_dir / 'data/equities_class.duckdb'), read_only=True) as con:
    experiment['dataset_metadata'] = dict(con.execute('SELECT key,value FROM dataset_metadata').fetchall())
display(pd.Series({k: v for k, v in experiment.items() if k not in ['features', 'dataset_metadata']}))
search = GridSearchCV(make_pipeline(Ridge()), {'model__alpha': experiment['alpha_grid']},
                      cv=folds, scoring='neg_root_mean_squared_error', n_jobs=1, error_score='raise')
search.fit(development[FEATURES], development.target)
experiment['chosen_alpha'] = search.best_params_['model__alpha']
print('Selected alpha:', experiment['chosen_alpha'])

## 2. Fit the model once

Let's fit our first model after the close on December 31, 2014. We can train on rows whose 90-day outcomes ended before that date. The model then keeps the same coefficients and fitted preprocessing throughout 2015.

We'll also make predictions for December 31 itself, because that signal determines the first January purchases. When producing predictions, we keep eligible stocks even if their future targets are missing. We know their features at the signal date; we don't yet know whether the dataset will contain an exit price three months later.

The model receives the explicit `FEATURES` list used during training. This keeps future-outcome columns such as `label_end` out of its inputs. Each prediction records a signal date, symbol, score, and fit date so we can trace it back to the model that made it.

In [ ]:
test_features = panel.loc[panel.eligible & panel.date.ge(fit_date)].copy()
fixed_model = search.best_estimator_
fixed_scores = test_features[['date', 'symbol']].copy()
fixed_scores['score'] = fixed_model.predict(test_features[FEATURES])
fixed_scores['fit_date'] = fit_date
assert development.label_end.max() < fixed_scores.date.min()
assert fixed_scores.fit_date.le(fixed_scores.date).all()
print('Scored rows:', len(fixed_scores), '; scored rows without observed targets:', test_features.target.isna().sum())
display(fixed_scores.head())

## 3. Refit as more returns become known

Now let's try updating the model before the January, April, July, and October purchases. This is a **walk-forward** experiment: each fit uses the information available at the time, then supplies forecasts until the next scheduled fit.

For example, imagine we're fitting after the close on June 30. Some signals from early 2015 have completed their holding periods, so we can add those returns to training. A signal from late June still has an unknown outcome. We enforce that distinction with `label_end < asof` at every fit.

We'll keep expanding the training window from 2010 rather than dropping older years. The Ridge penalty and feature definitions stay fixed, while the coefficients, imputation values, clipping bounds, and scaling statistics are fitted again. Using newly completed returns follows our original plan; choosing a new penalty because it improves the 2015 results would change the experiment after seeing the test.

The `fit_audit` table records how many rows each model used and when its last training outcome ended. It lets us check that training finished outcomes precede the signals that model scored.

In [ ]:
quarterly_parts, fit_audit = [], []
fit_signals = pd.to_datetime(experiment['quarterly_fit_signals'])
for i, asof in enumerate(fit_signals):
    next_asof = fit_signals[i+1] if i+1 < len(fit_signals) else pd.Timestamp('2016-01-01')
    available = panel.loc[panel.eligible & panel.date.ge('2010-01-01')
                          & panel.date.lt(asof) & panel.label_end.lt(asof)]
    to_score = panel.loc[panel.eligible & panel.date.ge(asof) & panel.date.lt(next_asof)]
    model = clone(fixed_model).fit(available[FEATURES], available.target)
    predicted = to_score[['date', 'symbol']].copy()
    predicted['score'] = model.predict(to_score[FEATURES])
    predicted['fit_date'] = asof
    quarterly_parts.append(predicted)
    fit_audit.append({'fit_date': asof, 'rows': len(available),
                      'last_training_signal': available.date.max(),
                      'last_label_end': available.label_end.max(),
                      'first_scored_signal': to_score.date.min(), 'last_scored_signal': to_score.date.max()})
    assert available.label_end.max() < to_score.date.min()
quarterly_scores = pd.concat(quarterly_parts, ignore_index=True)
fit_audit = pd.DataFrame(fit_audit)
assert not quarterly_scores.duplicated(['date', 'symbol']).any()
display(fit_audit)

## 4. Compare the strategies under the same trading rules

We now have forecasts from a fixed Ridge model and a quarterly-refitted Ridge model. Let's compare both with a 63-session momentum strategy. `MomentumStrategy` calculates its ranking from the historical prices passed into `generate_orders`, just as it did in the first notebook.

For Ridge, we'll use `PredictedScoreStrategy` to look up the predictions we prepared with our chosen fitting schedules. Its `score_stocks` method reads only the current signal date's rows. Keeping prediction generation separate here lets us inspect the training audit and compare forecast errors. Both strategy types inherit the same monthly schedule and equal allocation from `MonthlyTopNStrategy`.

Every strategy receives history through the signal close and a portfolio snapshot. The backtester advances the calendar, the OMS books the trades, and `BacktestAnalytics` reports the results. We create a fresh strategy and account for each run.

The S&P 500 total-return benchmark invests at the same first test close. It gives us a broad market comparison, while momentum helps us ask whether fitting Ridge improved stock selection under our particular portfolio rules.

```python
class PredictedScoreStrategy(MonthlyTopNStrategy):
    """Use model predictions prepared with the research notebook's fit schedule."""

    def __init__(self, scores, top_n=3, cohort_fraction=1 / 3, holding_days=90):
        super().__init__(top_n, cohort_fraction, holding_days)
        scores = scores[["date", "symbol", "score"]].copy()
        scores["date"] = pd.to_datetime(scores.date)
        if scores[["date", "symbol"]].isna().any().any():
            raise ValueError("Scores need dates and symbols")
        if scores.duplicated(["date", "symbol"]).any() or not np.isfinite(scores.score).all():
            raise ValueError("Scores must be finite with one row per date and symbol")
        self.scores = {date: group.set_index("symbol").score for date, group in scores.groupby("date")}

    def score_stocks(self, signal_date, execution_date, history):
        return self.scores.get(signal_date, pd.Series(dtype=float)).copy()
```

In [ ]:
strategy_settings = {
    'top_n': experiment['top_n'],
    'cohort_fraction': experiment['cohort_fraction'],
    'holding_days': experiment['holding_calendar_days'],
}
strategies = {
    'fixed Ridge': PredictedScoreStrategy(fixed_scores, **strategy_settings),
    'quarterly Ridge': PredictedScoreStrategy(quarterly_scores, **strategy_settings),
    '63-session momentum': MomentumStrategy(membership, lookback_sessions=63, **strategy_settings),
}
runs = {}
for name, strategy in strategies.items():
    oms = OrderManagementSystem(initial_cash=experiment['initial_cash'],
                                fee_bps=experiment['fee_bps_per_side'])
    runs[name] = Backtester(prices, benchmark, strategy, oms).run_backtest(
        experiment['test_start'], experiment['test_end'])
comparison = pd.DataFrame({name: BacktestAnalytics(run).summary(experiment['risk_free_hurdle']).strategy
                           for name, run in runs.items()})
comparison['S&P 500 total return'] = BacktestAnalytics(runs['fixed Ridge']).summary(
    experiment['risk_free_hurdle'])['S&P 500 total return']
display(comparison)

fig, ax = plt.subplots(figsize=(10, 5))
for name, run in runs.items():
    run.history.nav.div(experiment['initial_cash']).sub(1).plot(ax=ax, label=name)
runs['fixed Ridge'].history.benchmark_nav.div(experiment['initial_cash']).sub(1).plot(
    ax=ax, label='S&P 500 total return', color='black', ls='--')
ax.set(ylabel='Cumulative net return', title='2015: fixed Ridge, quarterly Ridge, and momentum')
ax.legend()
plt.tight_layout()

Suppose refitting changes the predicted returns but leaves the top three stocks unchanged. The strategies may make the same purchases and produce the same NAV. Even a different ranking may have little effect when most cash is tied up in older lots. Compare the rankings and fills alongside the return curves to see what the model actually changed.

### Evaluate the forecasts too

Portfolio results tell us what happened to our money. Prediction metrics tell us how closely the forecasts matched the observed 90-day returns. Both are useful, and they can move in different directions.

For a late-2015 signal, we may still own a position when the database ends. We include that position in final NAV, but we can't calculate its forecast error until its target is known. The report below uses completed targets and shows the last evaluated signal and label dates so we can see which part of the year it covers.

Because the 90-day targets overlap, we also can't compound their monthly averages as if they were consecutive monthly portfolio returns. The backtest's daily NAV is what accounts for the portfolio we actually hold.

In [ ]:
prediction_report = []
for name, scores in [('fixed Ridge', fixed_scores), ('quarterly Ridge', quarterly_scores)]:
    evaluation = test_features[['date', 'symbol', 'target', 'label_end']].merge(scores, on=['date', 'symbol'], validate='one_to_one')
    evaluation = evaluation.loc[evaluation.date.ge('2015-01-01') & evaluation.target.notna()]
    prediction_report.append({'model': name, 'last_labeled_signal': evaluation.date.max(),
                              'last_label_end': evaluation.label_end.max(),
                              **prediction_metrics(evaluation, evaluation.score).to_dict()})
display(pd.DataFrame(prediction_report))

## 5. Investigate what drove the results

Let's say the refitted model has a better rank IC but a lower final NAV. We can look at its fills to see whether it changed the purchased stocks, then use the analytics report to see how much cash was invested and how much trading cost. Changes among stocks we never bought won't help the portfolio, and a strong signal can't use money already committed to another lot.

The diagnostics report average cash usage, total fees, gross trading volume relative to average NAV, and open lots at the end. The OMS's PnL report separates realized gains and losses from those still in open positions.

We'll also rerun the fixed model at 0, 5, and 25 basis points per side. Costs reduce cash, which can change the quantities bought later, so their effect can extend beyond the fees deducted on the original trades. These runs show how sensitive the result is to our cost assumption; they don't measure actual market impact.

In [ ]:
diagnostics = pd.DataFrame({name: BacktestAnalytics(run).diagnostics()
                            for name, run in runs.items()}).T
display(diagnostics)
display(pd.DataFrame({name: run.pnl for name, run in runs.items()}))

cost_rows = []
for bps in [0, 5, 25]:
    strategy = PredictedScoreStrategy(fixed_scores, **strategy_settings)
    oms = OrderManagementSystem(initial_cash=experiment['initial_cash'], fee_bps=bps)
    run = Backtester(prices, benchmark, strategy, oms).run_backtest(
        experiment['test_start'], experiment['test_end'])
    metrics = BacktestAnalytics(run).summary(experiment['risk_free_hurdle']).strategy
    cost_rows.append({'fee_bps': bps, 'fees': run.pnl.total_fees, **metrics.to_dict()})
display(pd.DataFrame(cost_rows))

## 6. Save enough to reproduce the result

Imagine returning to this experiment a month from now and trying to explain a particular trade. We'd need the configuration, the model's score on that date, and the fill that followed. The next cell saves those records, along with validation results, fit dates, NAV, diagnostics, and PnL, under `Lectures/modeling/outputs/class_research/`.

That directory is ignored by Git. For HW4, commit the small summaries and figures needed to explain your findings, and include instructions for regenerating the larger files.

In [ ]:
output_dir = modeling_dir / 'outputs/class_research'
output_dir.mkdir(parents=True, exist_ok=True)
(output_dir / 'experiment.json').write_text(json.dumps(experiment, indent=2))
pd.DataFrame(search.cv_results_).to_csv(output_dir / 'cv_results.csv', index=False)
fit_audit.to_csv(output_dir / 'fit_audit.csv', index=False)
comparison.to_csv(output_dir / 'performance.csv')
for name, run in runs.items():
    stem = name.replace(' ', '_')
    run.history.to_csv(output_dir / f'{stem}_nav.csv')
    run.fills.to_csv(output_dir / f'{stem}_fills.csv', index=False)
    run.open_lots.to_csv(output_dir / f'{stem}_open_lots.csv', index=False)
    run.diagnostics.to_csv(output_dir / f'{stem}_diagnostics.csv', index=False)
    run.pnl.to_csv(output_dir / f'{stem}_pnl.csv')
fixed_scores.to_csv(output_dir / 'fixed_predictions.csv', index=False)
quarterly_scores.to_csv(output_dir / 'quarterly_predictions.csv', index=False)
print('Saved experiment, validation, prediction, fit-audit, and accounting tables to:', output_dir)

## 7. Take the experiment to the homework dataset

Once the class example works, we can move to `equities_top2000.duckdb`. It contains 3,849 stocks selected across quarterly universes, including 1,355 marked inactive in the source snapshot. Use `universe_membership` to find the candidates for each entry quarter, and keep price history for anything already held. The [assignment](hw4.md) and [data guide](../modeling/README.md) explain the dataset and its limitations.

The larger file changes how we load and prepare data. Read only the columns and dates needed, and calculate per-stock features in batches with enough warm-up history. If we sample training rows at month-end, we still calculate daily features first. Each entry signal needs the full eligible cross-section, and the portfolio still needs daily prices for its holdings.

Time data loading, feature calculation, fitting, and simulation separately before trying to speed things up. Start with a small model search. The homework test ends on September 30, 2026, with open positions included in NAV. If a held stock loses price coverage, investigate it under your stated missing-price policy; the dataset's incomplete delisting records don't tell us what payout to assume.

## Exercises

### Your turn: reconstruct the cash balance

Imagine you're reviewing the fixed Ridge run and want to check the OMS independently. Starting with $1,000,000, add the cumulative daily `cash_change` from its fills and align the result to every history date. Store that Series in `your_reconstructed_cash`, then compare it with the account's cash column.

In [ ]:
your_reconstructed_cash = None

In [ ]:
if your_reconstructed_cash is None:
    print('Group fill cash changes by date, reindex missing days to zero, then cumulatively sum.')
else:
    h = runs['fixed Ridge'].history
    pd.testing.assert_index_equal(your_reconstructed_cash.index, h.index)
    np.testing.assert_allclose(your_reconstructed_cash, h.cash, atol=1e-7)
    print('Independent cash reconstruction passed.')

### Your turn: audit the June fit

Let's return to the June 30, 2015 fit. Count eligible rows since 2010 whose signals came before that fit but whose labels end on or after it, or are unknown. Store the count in `your_immature_rows`. In `your_retraining_note`, explain why we can't train on those rows yet, even though their features already existed.

In [ ]:
your_immature_rows = None
your_retraining_note = ''

In [ ]:
if your_immature_rows is None:
    print('An observed feature row can still have an unobserved target.')
else:
    cutoff = pd.Timestamp('2015-06-30')
    candidates = panel.loc[panel.eligible & panel.date.ge('2010-01-01') & panel.date.lt(cutoff)]
    expected = (candidates.label_end.ge(cutoff) | candidates.label_end.isna()).sum()
    assert your_immature_rows == expected and your_retraining_note.strip()
    print('Retraining audit passed. Every fit needs this same maturity check.')

### Your turn: write the research conclusion

Write 150–250 words explaining what you learned from the forecast and portfolio results. Include any disappointing result, identify one data limitation that code can't repair, and propose the next experiment to try on development data.